In [74]:
import numpy as np
import pandas as pd

In [75]:
data= pd.read_csv("dataset.csv")

In [76]:
# Subtask 1
answer1= pd.DataFrame([{
    'subtaskID': 1,
    'Value1': len(data['id'].unique()),
    'Value2': len(data['vehicle_type'].unique())
}])

answer1.head()

,subtaskID,Value1,Value2
0,1,663,4


In [77]:
# Subtask 2
from sklearn.cluster import DBSCAN

positions = data.groupby("id")[["latitude", "longitude"]].mean().reset_index()
data2= np.radians(positions[['latitude', 'longitude']].values)

kms_per_radian = 6370
eps_km = 7 
eps = eps_km / kms_per_radian

model= DBSCAN(eps= eps, metric='haversine')

predictions= model.fit_predict(data2)

answer2= pd.DataFrame([{
    'subtaskID': 2,
    'Value1': id_,
    'Value2': pred
} for id_, pred in zip(positions['id'], predictions)])

answer2.head()

,subtaskID,Value1,Value2
0,2,0,0
1,2,1,0
2,2,2,0
3,2,3,0
4,2,4,0


In [78]:
# Subtask 3
from sklearn.cluster import KMeans

data["timestamp"] = pd.to_datetime(data["timestamp"])

df = data[(data["vehicle_type"] == 10) &
          ((data["timestamp"].dt.hour >= 23) | (data["timestamp"].dt.hour <= 5))].copy()

df["lat_diff"] = df.groupby("id")["latitude"].diff().abs()
df["lon_diff"] = df.groupby("id")["longitude"].diff().abs()

df_stationary = df[(df["lat_diff"].fillna(0) < 0.0001) &
                   (df["lon_diff"].fillna(0) < 0.0001)]

positions = df_stationary[["latitude", "longitude"]].values

model = KMeans(n_clusters=3, random_state=42)
labels = model.fit_predict(positions)

centers = []
for i in range(3):
    cluster_points = positions[labels == i]
    centers.append([
        np.median(cluster_points[:, 0]),
        np.median(cluster_points[:, 1])
    ])

centers = sorted(centers, key=lambda x: x[0])

answer3 = pd.DataFrame([{
    'subtaskID': 3,
    'Value1': center[0],
    'Value2': center[1]
} for center in centers])

answer3.head()


,subtaskID,Value1,Value2
0,3,64.8632,16.4329
1,3,64.9051,16.4384
2,3,64.9108,16.4011


In [79]:
answer= pd.concat([answer1, answer2, answer3])
answer.to_csv("submission.csv", index= False)